In [3]:
import json                                        # For reading/writing JSON files
import re                                          # Regular expressions for text splitting
import os                                          # For creating directories

os.makedirs("chunks", exist_ok=True)               # Create the "chunks" output folder if it doesn't exist

with open("raw_laws/alle_gesetze.json", "r", encoding="utf-8") as f:  # Open the combined law file created by download_full.ipynb
    data = json.load(f)                            # Parse the JSON into a Python list of paragraph dicts

def clean_text(text):                              # Function to strip the metadata header from the raw paragraph text
    """
    Entfernt den Metadaten-Header (Kurztitel, BGBl. etc.)
    Der echte Gesetzestext beginnt erst nach diesen Infos.
    """
    zeilen = text.splitlines()                     # Split the full text into individual lines
    
    meta_keywords = [                              # List of keywords that appear only in the metadata header, not actual law text
        "Kurztitel", "Kundmachungsorgan", "Bundesgesetzblatt",
        "Typ", "§/Artikel/Anlage", "Paragraph/Artikel/Anlage",
        "Inkrafttretensdatum", "Abkürzung", "Index", "Beachte",
        "zum Bezugszeitraum", "BGBl.", "BG", "32/02",
        "Einkommensteuergesetz", "Umsatzsteuergesetz",
        "Körperschaftsteuergesetz", "Grunderwerbsteuergesetz",
    ]
    
    start_index = 0                                # Default: start from the very beginning
    for i, zeile in enumerate(zeilen):             # Loop through each line with its index
        ist_meta = any(keyword in zeile for keyword in meta_keywords)  # True if the line contains any metadata keyword
        ist_kurz = len(zeile.strip()) < 4          # True if the line is too short to be real content (< 4 characters)
        if not ist_meta and not ist_kurz and i >= 5:  # If the line is past the first 5 lines AND not metadata AND not too short...
            start_index = i                        # Mark this as the start of the actual law text
            break                                  # Stop searching; we found where the real content begins
    
    return "\n".join(zeilen[start_index:]).strip() # Join all lines from the start index onward and remove surrounding whitespace


def split_into_chunks(text, paragraph_info, max_zeichen=800):  # Function to split one paragraph's text into smaller chunks
    """
    Teilt Text zweistufig in Chunks auf:
    Stufe 1: An Absatzgrenzen aufteilen wo "(1)" "(2)" steht
    Stufe 2: Wenn Absatz immer noch zu lang -> an Satzgrenzen aufteilen
    """
    chunks = []                                    # List to collect all chunk dicts for this paragraph
    chunk_nummer = 1                               # Counter to number chunks within the paragraph
    
    # Stufe 1: An "(1)", "(2)" etc. aufteilen
    absaetze = re.split(r'\n(?=\(\d+\))', text)   # Split the text at every newline that is followed by "(1)", "(2)", etc.
    
    for absatz in absaetze:                        # Loop through each section split at the numbered paragraph markers
        absatz = absatz.strip()                    # Remove leading/trailing whitespace from this section
        if not absatz:                             # Skip empty sections
            continue
        
        # Stufe 2: Absatz selbst noch zu lang -> an Sätzen aufteilen
        if len(absatz) > max_zeichen:              # If this section is still too long to be a single chunk...
            saetze = re.split(r'(?<=\.)\s+', absatz)  # Split at sentence boundaries (after a period followed by whitespace)
            aktueller_chunk = ""                   # Initialize an empty accumulator for the current chunk
            
            for satz in saetze:                    # Loop through each sentence
                if len(aktueller_chunk) + len(satz) > max_zeichen and aktueller_chunk:  # If adding this sentence would exceed the limit AND we already have content...
                    chunks.append({                # Save the current chunk as a complete entry
                        "gesetz": paragraph_info['gesetz'],        # Law name
                        "paragraph": paragraph_info['paragraph'],  # Paragraph label
                        "chunk_nr": chunk_nummer,                  # Sequential chunk number within this paragraph
                        "quelle": paragraph_info['quelle'],        # Source citation string
                        "text": aktueller_chunk.strip()            # The accumulated chunk text
                    })
                    chunk_nummer += 1              # Increment the chunk counter
                    aktueller_chunk = satz         # Start a new chunk with the current sentence
                else:
                    aktueller_chunk += " " + satz  # Otherwise, just add the sentence to the current chunk
            
            # Letzten Teil nicht vergessen
            if aktueller_chunk.strip():            # If there's remaining content after the last sentence...
                chunks.append({                    # Save it as the final chunk
                    "gesetz": paragraph_info['gesetz'],
                    "paragraph": paragraph_info['paragraph'],
                    "chunk_nr": chunk_nummer,
                    "quelle": paragraph_info['quelle'],
                    "text": aktueller_chunk.strip()
                })
                chunk_nummer += 1                  # Increment chunk counter
        
        else:
            # Absatz kurz genug -> direkt als Chunk speichern
            chunks.append({                        # The section fits within the limit, save it directly as one chunk
                "gesetz": paragraph_info['gesetz'],
                "paragraph": paragraph_info['paragraph'],
                "chunk_nr": chunk_nummer,
                "quelle": paragraph_info['quelle'],
                "text": absatz
            })
            chunk_nummer += 1                      # Increment chunk counter
    
    return chunks                                  # Return the list of all chunks for this paragraph


# Alle Paragraphen verarbeiten
alle_chunks = []                                   # Master list to collect all chunks across all laws and paragraphs

for eintrag in data:                               # Loop through every paragraph in the dataset
    
    # § 0 überspringen - das ist nur der Metadaten-Header des Gesetzes
    if eintrag['paragraphnummer'] == '0':          # Skip paragraph "0" — it's just the law's title/header block
        continue
    
    text_sauber = clean_text(eintrag['text'])      # Strip the metadata header from this paragraph's text
    if not text_sauber:                            # If nothing remains after cleaning, skip this entry
        continue
    
    chunks = split_into_chunks(text_sauber, eintrag)  # Split the cleaned text into chunks, passing the paragraph metadata
    alle_chunks.extend(chunks)                     # Add all resulting chunks to the master list

# Zu kurze Chunks rausfiltern (unter 50 Zeichen - das ist kein sinnvoller Inhalt)
alle_chunks = [c for c in alle_chunks if len(c['text']) >= 50]  # Remove any chunks shorter than 50 characters — not meaningful content

# Speichern - überschreibt die alte alle_chunks.json
with open("chunks/alle_chunks.json", "w", encoding="utf-8") as f:  # Open the output file for writing
    json.dump(alle_chunks, f, ensure_ascii=False, indent=2)         # Save all chunks as pretty-printed JSON, preserving German characters

# Statistik
laengen = [len(c['text']) for c in alle_chunks]   # Compute the character length of every chunk
print(f"Gesamt Chunks:   {len(alle_chunks)}")      # Print total number of chunks
print(f"Längster Chunk:  {max(laengen)} Zeichen")  # Print the longest chunk's length
print(f"Kürzester Chunk: {min(laengen)} Zeichen")  # Print the shortest chunk's length
print(f"Durchschnitt:    {int(sum(laengen)/len(laengen))} Zeichen")  # Print the average chunk length

from collections import Counter                    # Import Counter for grouping
for gesetz, anzahl in Counter(c['gesetz'] for c in alle_chunks).items():  # Count chunks per law
    print(f"  {gesetz}: {anzahl} Chunks")          # Print chunk count for each law

print(f"\n=== Beispiel-Chunk ===")                 # Section header for the sample output
print(f"Quelle: {alle_chunks[20]['quelle']}")      # Print the source citation of chunk #20
print(f"Länge: {len(alle_chunks[20]['text'])} Zeichen")  # Print the length of chunk #20
print(f"Text:\n{alle_chunks[20]['text']}")         # Print the full text of chunk #20

Gesamt Chunks:   6685
Längster Chunk:  1892 Zeichen
Kürzester Chunk: 56 Zeichen
Durchschnitt:    587 Zeichen
  EStG 1988: 4280 Chunks
  UStG 1994: 1188 Chunks
  KStG 1988: 897 Chunks
  GrEStG 1987: 320 Chunks

=== Beispiel-Chunk ===
Quelle: EStG 1988 § 2
Länge: 304 Zeichen
Text:
Buchführende Land- und Forstwirte und rechnungslegungspflichtige Gewerbetreibende (Paragraph 5,) dürfen jedoch ein vom Kalenderjahr abweichendes Wirtschaftsjahr haben; in diesem Fall ist der Gewinn bei Ermittlung des Einkommens für jenes Kalenderjahr zu berücksichtigen, in dem das Wirtschaftsjahr endet.
